# PEFT LoRA 指令微調 Chatbot（2026 版）

本 notebook 以 **LoRA（Low-Rank Adaptation）** 對中文因果語言模型進行指令微調，目標是讓基礎模型能理解並回應 Alpaca 格式的問答指令。

## 學習目標

1. 理解 LoRA 如何以極少可訓練參數達到接近全量微調的效果
2. 掌握 2026 年的標準 SFT（Supervised Fine-Tuning）流程：`SFTTrainer` + `SFTConfig` + `LoraConfig`
3. 學會用 `tokenizer.apply_chat_template()` 取代手寫 prompt 格式，確保訓練/推論一致
4. 理解為何使用 `bf16`、`device_map='auto'`、`safetensors` 以及 `-100` 標籤遮罩的原理

## 前置知識

- 了解 Transformer 架構（注意力機制）
- 了解因果語言模型（Causal LM）的預訓練目標

## 與相鄰 notebook 的銜接

- 上一個：[../00-concepts/lora_concepts.ipynb](../00-concepts/lora_concepts.ipynb)（LoRA 數學原理）
- 下一個：[../02-QLoRA/chatbot_qlora.ipynb](../02-QLoRA/chatbot_qlora.ipynb)（量化 + LoRA，記憶體再減半）

## 硬體需求

| 設定 | 最低 VRAM |
|------|-----------|
| `bloom-1b4` + LoRA，bf16 | ~4 GB |
| `bloom-1b4` + LoRA，4-bit 量化（QLoRA） | ~2 GB |

In [ ]:
# ── 版本鎖定（2026 統一慣例）──────────────────────────────────────────
# 確保執行環境與課程測試版本一致，避免 API 不相容
# 如果是全新環境，取消下方的 pip install 的 # 號

# !pip install -q \
#   "transformers>=4.46" \
#   "datasets>=3.0" \
#   "trl>=0.12" \
#   "peft>=0.13" \
#   "accelerate>=1.0" \
#   "bitsandbytes>=0.44" \
#   "evaluate>=0.4" \
#   "safetensors>=0.4" \
#   "torch>=2.4"

import importlib, sys
for pkg, attr in [
    ("transformers", "__version__"),
    ("datasets",     "__version__"),
    ("trl",          "__version__"),
    ("peft",         "__version__"),
    ("accelerate",   "__version__"),
]:
    mod = importlib.import_module(pkg)
    print(f"{pkg:>15}: {getattr(mod, attr)}")

## Step 1 — 匯入套件與環境確認

`set_seed(42)` 確保資料抽樣、初始化、dropout 等隨機行為在不同次執行間保持一致，是可重現性的最低門檻。

In [ ]:
import torch
from transformers import set_seed

set_seed(42)

cuda_available = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
print(f"CUDA available : {cuda_available}")
print(f"Device         : {device_name}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig

## Step 2 — 載入資料集

[c-s-ale/alpaca-gpt4-data-zh](https://huggingface.co/datasets/c-s-ale/alpaca-gpt4-data-zh) 是繁/簡中文版的 Alpaca-GPT4 指令資料集，每筆包含：

- `instruction`：任務說明
- `input`：（可選）補充脈絡
- `output`：模型應產生的回應

這裡只取前 50% 以縮短訓練時間；實際課程外練習可移除 `split` 限制。

In [ ]:
ds = load_dataset("c-s-ale/alpaca-gpt4-data-zh", split="train[:50%]")
print(ds)
print("\n第一筆範例：")
print(ds[0])

## Step 3 — 載入 Tokenizer

**為何 bfloat16 優於 float16？**

- bf16 與 fp32 共享相同的 8-bit 指數範圍，**不會發生梯度下溢（underflow）**，訓練穩定性遠優於 fp16
- fp16 的 5-bit 指數讓梯度容易爆炸或消失，需要額外的 loss scaling 機制
- Ampere（A100）/ Ada Lovelace 及以上 GPU 對 bf16 有硬體加速，速度與 fp16 相當

**為何 safetensors 優於 pickle？**

- pickle 本質上允許執行任意 Python 程式碼，下載陌生模型有安全風險
- safetensors 只儲存 tensor 資料，載入速度快 3–10 倍（免 Python deserialization overhead）

**為何 `device_map='auto'` 優於 `.cuda()`？**

- `device_map='auto'` 讓 accelerate 自動決定每一層的放置位置：GPU VRAM 足夠時全部放 GPU；不足時 offload 到 CPU RAM；RAM 也不足時 offload 到磁碟。
- `model.cuda()` 強制所有層放 GPU，VRAM 不足就直接崩潰。

In [ ]:
MODEL_ID = "Langboat/bloom-1b4-zh"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 若 tokenizer 沒有 chat_template，補上最小可用模板（Alpaca 格式）
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{% if message['role'] == 'user' %}Human: {{ message['content'] }}\n\nAssistant: "
        "{% elif message['role'] == 'assistant' %}{{ message['content'] }}{{ eos_token }}"
        "{% endif %}"
        "{% endfor %}"
    )
    print("[INFO] chat_template was None — applied Alpaca-style fallback template.")

print(tokenizer)

## Step 4 — 理解 apply_chat_template（WHY）

`tokenizer.apply_chat_template()` 將格式邏輯封裝在 tokenizer 的 `chat_template` 欄位（Jinja2 模板），使用方只需傳入標準的 `messages` 列表：

```python
messages = [
    {"role": "user",      "content": "如何有效複習？"},
    {"role": "assistant", "content": "可以用番茄鐘法..."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

訓練與推論都走同一條路徑，確保 prompt 格式完全一致。此機制在 2026 的多模態模型（image/audio token）中完全相容——這是通往 05-Multimodal 的關鍵橋樑。

下面示範如何把 Alpaca 格式的欄位組成 `messages` 列表，再用 `apply_chat_template` 轉換：

In [ ]:
def alpaca_to_messages(example: dict) -> dict:
    """
    Convert Alpaca-format fields into the standard 'messages' list format.
    Merges 'instruction' and optional 'input' into the user turn.
    """
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = f"{example['instruction']}\n{example['input']}"
    return {
        "messages": [
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": example["output"]},
        ]
    }

# 示範一筆
sample = alpaca_to_messages(ds[0])
full_text = tokenizer.apply_chat_template(
    sample["messages"], tokenize=False, add_generation_prompt=False
)
print("=== 組裝後的完整文字 ===")
print(full_text)

## Step 5 — 資料集預處理（手動版，用於理解底層原理）

> **注意**：以下手刻版本是為了讓你理解「-100 遮罩的原理」；實際訓練我們會用 `SFTTrainer` 自動完成這一步（見 Step 7）。

### 為什麼 labels 要用 -100？

Causal LM 的訓練目標是 next-token prediction（預測下一個 token）。在指令微調時，我們**只希望模型學習如何生成回應（assistant 部分），而不是學習把人類的問題重複出來**。

PyTorch 的 `CrossEntropyLoss` 預設會忽略 `ignore_index=-100` 的位置，因此：

```
 input_ids : [Human: 考試技巧？<sep>  可以用番茄鐘法...  <eos>]
 labels    : [  -100  -100  -100   可以用番茄鐘法...  <eos>]
                 ^── instruction 部分設為 -100，不計算 loss
```

這確保模型只在「回應」部分被懲罰，不被迫記憶 prompt 格式。

In [ ]:
def process_func_manual(example: dict) -> dict:
    """
    Manual tokenization with -100 label masking for the instruction part.
    This is the 'bottom-up' reference implementation; SFTTrainer handles this automatically.
    """
    MAX_LENGTH = 256

    # Build messages and split at the boundary
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = f"{example['instruction']}\n{example['input']}"

    # Tokenize the instruction (prompt) part — with add_generation_prompt=True
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False,
        add_generation_prompt=True,  # appends 'Assistant: ' so model knows to continue
    )
    # Tokenize the response part (includes eos_token)
    response_text = example["output"] + tokenizer.eos_token

    prompt_ids   = tokenizer(prompt_text,    add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response_text,  add_special_tokens=False)["input_ids"]

    input_ids      = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)
    labels         = [-100] * len(prompt_ids) + response_ids  # mask the prompt part

    # Truncate to MAX_LENGTH
    input_ids      = input_ids[:MAX_LENGTH]
    attention_mask = attention_mask[:MAX_LENGTH]
    labels         = labels[:MAX_LENGTH]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


# Use batched=True for 3-5x speed improvement over row-by-row processing
tokenized_ds = ds.map(
    process_func_manual,
    remove_columns=ds.column_names,
    batched=False,   # single-example mode here so logic stays readable
    desc="Tokenizing dataset",
)
print(tokenized_ds)

In [ ]:
# Verify: decode input_ids and non-(-100) labels for one example
print("=== input_ids decoded ===")
print(tokenizer.decode(tokenized_ds[1]["input_ids"]))
print("\n=== labels decoded (only response tokens) ===")
print(tokenizer.decode([t for t in tokenized_ds[1]["labels"] if t != -100]))

## Step 6 — 載入基礎模型（2026 統一慣例）

### device_map='auto' 的語意

`device_map='auto'` 讓 accelerate 自動決定每一層的放置位置：

1. **GPU VRAM 足夠**：全部放 GPU
2. **VRAM 不夠但 RAM 足夠**：超出部分 offload 到 CPU RAM（速度變慢但不 OOM）
3. **RAM 也不夠**：超出部分 offload 到磁碟（極慢，但至少能跑）

相比之下，`model.cuda()` 強制所有層放 GPU，VRAM 不足就直接崩潰。

In [ ]:
# VRAM estimate: bloom-1b4 in bf16 ≈ 2.8 GB
# If your GPU has < 4 GB VRAM, consider using QLoRA (see 02-QLoRA notebook)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",           # replaces model.cuda() / device=0
    torch_dtype=torch.bfloat16, # bf16: stable gradients, same speed as fp16 on Ampere+
    use_safetensors=True,        # safe, 3-10x faster load than pickle .bin files
)

print(f"Model dtype  : {model.dtype}")
print(f"Model device : {next(model.parameters()).device}")
print(f"Total params : {sum(p.numel() for p in model.parameters()):,}")

### 觀察模型結構

執行下方 cell 可以看到所有 layer 名稱，幫助你判斷 LoRA 應該 target 哪些模組。`bloom-1b4` 的注意力模組名稱是 `query_key_value`，我們稍後會 target 它。

In [ ]:
# Print all named parameters to identify target modules for LoRA
for name, param in model.named_parameters():
    print(f"{name:60s}  {str(param.shape):30s}  requires_grad={param.requires_grad}")

## Step 7 — 建立 LoRA 設定

### LoRA 的核心思想

全量微調會更新所有 W（權重矩陣），記憶體開銷巨大。LoRA 認為「微調帶來的權重更新矩陣 ΔW 具有低秩結構」，因此改為訓練兩個小矩陣 A 和 B，使得 ΔW = BA（rank << 原始維度）。

推論時 ΔW = BA 被合併（merge）回原始 W，零額外延遲。

### LoraConfig 參數說明

| 參數 | 值 | 說明 |
|------|-----|------|
| `r` | 8 | LoRA 秩（rank），越大 ≈ 越多可訓練參數；常見值：4/8/16/32 |
| `lora_alpha` | 32 | 縮放係數，有效學習率 ≈ `alpha/r` ，通常設 `2*r` |
| `target_modules` | `["query_key_value"]` | 只替換注意力的 QKV 投影，是 bloom 的標準做法 |
| `lora_dropout` | 0.1 | 防止 LoRA adapter 過擬合 |
| `bias` | `"none"` | 不訓練 bias，節省記憶體 |
| `modules_to_save` | `["word_embeddings"]` | 這些模組**全量**更新（而非 LoRA），因為 embedding 影響輸出空間 |

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # bloom-specific: fused QKV projection
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["word_embeddings"],  # full update for embeddings
)
print(lora_config)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params << total params, e.g. ~0.3% for r=8

In [ ]:
# Inspect the renamed parameter tree after PEFT wrapping
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name:70s}  {str(param.shape)}")

## Step 8 — 配置訓練參數（SFTConfig / TrainingArguments）

### 關鍵參數說明

| 參數 | 原因 |
|---------|------|
| `bf16=True` | 確保訓練在 bf16 精度進行，與模型載入精度一致 |
| `warmup_ratio=0.1` | 前 10% 步驟線性暖機，防止初始大梯度破壞 LoRA adapter |
| `lr_scheduler_type='cosine'` | cosine 衰減在 LLM 微調中普遍優於 linear |
| `max_grad_norm=1.0` | 梯度裁剪，防止梯度爆炸 |
| `optim='adamw_torch_fused'` | fused kernel，比標準 AdamW 快 15-25%（PyTorch 2.0+） |
| `save_safetensors=True` | checkpoint 儲存為 safetensors 格式 |
| `seed=42` | 確保 DataLoader shuffle 的可重現性 |

**Effective batch size = per_device_train_batch_size × gradient_accumulation_steps**

本例：8 × 16 = 128 個樣本才更新一次參數，等效於大 batch 訓練，卻只佔用小 batch 的 VRAM。

**為何用 AdamW 而不是 Adam？**

Adam 的 L2 正則化（weight decay）與自適應學習率的交互作用會讓正則化強度因參數的梯度大小而異，造成不一致的正則化效果。AdamW 把 weight decay 從梯度更新中分離，正則化更乾淨，收斂性更好。

In [ ]:
sft_config = SFTConfig(
    output_dir="./chatbot_lora_2026",
    # --- batch & gradient ---
    per_device_train_batch_size=8,
    gradient_accumulation_steps=16,  # effective batch = 8 * 16 = 128
    # --- epochs & steps ---
    num_train_epochs=1,
    # --- precision ---
    bf16=True,                        # match model dtype; stable vs fp16
    # --- optimizer ---
    optim="adamw_torch_fused",        # fused AdamW, ~20% faster on PyTorch 2.4+
    learning_rate=2e-4,
    max_grad_norm=1.0,
    # --- lr schedule ---
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    # --- logging & saving ---
    logging_steps=20,
    save_steps=100,
    save_total_limit=2,
    save_safetensors=True,            # save checkpoints as safetensors
    # --- reproducibility ---
    seed=42,
    # --- SFTTrainer-specific ---
    max_seq_length=256,               # replaces MAX_LENGTH in manual process_func
    packing=False,                    # set True to pack multiple short examples into one sequence
)
print(sft_config)

## Step 9 — 建立 SFTTrainer（2026 主要路徑）

`SFTTrainer` 是 TRL 提供的 `Trainer` 子類別，專為指令微調設計，自動處理：

1. **格式化**：透過 `formatting_func` 或 `dataset_text_field` 將資料轉換為文字
2. **-100 遮罩**：如果指定 `response_template`，自動只對 response 部分計算 loss（`DataCollatorForCompletionOnlyLM`）
3. **Sequence packing**：可選，把多個短樣本打包成一個序列，提升 GPU 利用率
4. **PEFT 整合**：透過 `peft_config` 一鍵注入 LoRA，不需要手動呼叫 `get_peft_model`

> 本 notebook 為了教學對照，已在 Step 5 手動呼叫 `get_peft_model`。在實際工程中，可以直接把 `lora_config` 傳給 `SFTTrainer(peft_config=lora_config)` 並省略 Step 7 的 `get_peft_model`。

In [ ]:
def formatting_func(example: dict) -> str:
    """
    Converts an Alpaca-format example to a single string using apply_chat_template.
    SFTTrainer calls this per example (or per batch if batched=True).

    Using apply_chat_template ensures training and inference use identical formatting,
    eliminating the #1 source of train/inference mismatch in older codebases.
    """
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = f"{example['instruction']}\n{example['input']}"

    messages = [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # False during training: label includes full response
    )


# Sanity check
print(formatting_func(ds[0]))

In [ ]:
# Re-load a fresh base model for the SFTTrainer path
# (the previous model was already wrapped with get_peft_model for the manual demo)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)

trainer = SFTTrainer(
    model=base_model,
    args=sft_config,
    train_dataset=ds,
    peft_config=lora_config,         # SFTTrainer injects LoRA automatically
    formatting_func=formatting_func, # apply_chat_template inside
    processing_class=tokenizer,      # 2026: use 'processing_class' instead of 'tokenizer'
)

## Step 10 — 模型訓練

In [ ]:
trainer.train()

## Step 11 — 儲存模型（safetensors 格式）

`safe_serialization=True` 確保 adapter 以 safetensors 格式儲存，而非 pickle。

`push_to_hub()` 可以選擇性地把 adapter 上傳到 Hugging Face Hub，方便後續部署。若不需要上傳，執行第一個 cell 即可。

In [ ]:
ADAPTER_SAVE_PATH = "./chatbot_lora_adapter"

trainer.model.save_pretrained(
    ADAPTER_SAVE_PATH,
    safe_serialization=True,  # saves as .safetensors, not .bin (pickle)
)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"Adapter saved to: {ADAPTER_SAVE_PATH}")

In [ ]:
# Optional: push adapter to Hugging Face Hub
# Uncomment and fill in your Hub repo name

# from huggingface_hub import HfApi
# trainer.model.push_to_hub(
#     "your-username/bloom-1b4-zh-chatbot-lora",
#     private=True,
# )
# tokenizer.push_to_hub(
#     "your-username/bloom-1b4-zh-chatbot-lora",
#     private=True,
# )

## Step 12 — 模型推理

推論時使用 `apply_chat_template` 確保 prompt 格式與訓練完全一致，並設定 `add_generation_prompt=True`，讓模板在結尾加上觸發符號引導模型生成回應。`device_map='auto'` 已在 `from_pretrained` 時設定，推論時不需要手動搬移張量至特定裝置。

In [ ]:
from peft import PeftModel

# Load the LoRA adapter on top of the base model
# device_map='auto' handles placement; no .cuda() needed
inference_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
inference_model = PeftModel.from_pretrained(inference_model, ADAPTER_SAVE_PATH)
inference_model.eval()

In [ ]:
def chat(user_message: str, max_new_tokens: int = 128) -> str:
    """
    Generate a response using the fine-tuned LoRA model.
    Uses apply_chat_template to ensure format consistency with training.
    """
    messages = [{"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # append 'Assistant: ' trigger for inference
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)

    with torch.inference_mode():  # faster than torch.no_grad() for inference
        output_ids = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,  # avoid warning if pad_token not set
        )

    # Decode only the newly generated tokens (exclude the prompt)
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# Test
questions = [
    "考試有哪些技巧？",
    "如何有效地學習一門新語言？",
    "請介紹一個健康的早晨習慣",
]

for q in questions:
    print(f"User     : {q}")
    print(f"Assistant: {chat(q)}")
    print("-" * 60)

## Step 13 — 合併 LoRA Adapter 至 Base Model（選用）

如果後續要部署，可以把 LoRA 的 ΔW = BA 合併回原始 W，消除推論時的額外計算。合併後的模型就是一個普通的 `AutoModelForCausalLM`，不再需要 PEFT 套件。

> **注意**：合併後模型無法還原 adapter，請確保已保存 adapter。合併後 VRAM 需求等同於基礎模型（bf16 約 2.8 GB）。

In [ ]:
# Optional: merge LoRA weights into base model for deployment
# VRAM requirement: same as base model in bf16 (~2.8 GB for bloom-1b4)

merged_model = inference_model.merge_and_unload()  # returns a plain AutoModelForCausalLM

MERGED_SAVE_PATH = "./chatbot_lora_merged"
merged_model.save_pretrained(
    MERGED_SAVE_PATH,
    safe_serialization=True,
)
tokenizer.save_pretrained(MERGED_SAVE_PATH)
print(f"Merged model saved to: {MERGED_SAVE_PATH}")

## 小結

本 notebook 示範了 LoRA 指令微調的完整流程，涵蓋以下環節：

- **模型載入**：`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True`
- **Prompt 格式**：`tokenizer.apply_chat_template(messages, ...)` 確保訓練與推論格式一致
- **-100 遮罩**：只對 assistant 回應部分計算 loss，不懲罰 prompt 重複
- **SFTTrainer**：自動處理格式化、遮罩、PEFT 注入
- **訓練精度**：`bf16=True`，cosine schedule + warmup，AdamW fused optimizer
- **儲存格式**：`safe_serialization=True`（safetensors）

## 練習

1. **調整 LoRA 秩**：把 `r` 從 8 改成 4 和 16，觀察 `print_trainable_parameters()` 的輸出與訓練時間的變化。
2. **Target 更多模組**：在 `LoraConfig` 中加入 `dense`、`dense_h_to_4h`、`dense_4h_to_h`，觀察效果差異。
3. **開啟 packing**：把 `SFTConfig` 中的 `packing=True`，觀察每個 step 的樣本利用率。
4. **換模型**：把 `MODEL_ID` 改為 `Qwen/Qwen2.5-0.5B-Instruct`，`target_modules` 改為 `["q_proj", "v_proj"]`，觀察 `chat_template` 是否已內建（不需要手動設定 fallback）。
5. **推論一致性測試**：在推論時移除 `apply_chat_template`，改用手寫格式，觀察輸出品質下降幅度，體會訓練／推論格式不一致的代價。

## 下一步

→ [../02-QLoRA/chatbot_qlora.ipynb](../02-QLoRA/chatbot_qlora.ipynb)：加入 `BitsAndBytesConfig` 4-bit 量化，讓 VRAM 需求再減半，在消費級 GPU 上訓練 7B 以上的模型。